# AloePri — Qwen3-8B obfusqué via une interface OpenAI-compatible

Variante du notebook qui interroge le modèle à travers le **proxy local OpenAI-compatible** (`aloepri_modal/openai_proxy.py`, port 8001) : on utilise le client `openai` officiel comme si le modèle était un endpoint OpenAI standard.

- Le proxy local fait la **permutation côté client** (clés + tokenizer sur cette machine) puis appelle `POST /generate` de Modal avec des **IDs permutés** — Modal ne voit jamais de texte ni les clés.
- Mêmes réglages de décodage validés (non-thinking, greedy, pénalité de répétition, blocage `<think>`).

In [ ]:
# --- Configuration + démarrage du proxy local (une seule fois) ---
KEYS_PATH = "/home/cmauceri/deepseek-harness-ws/artifacts/obfuscation_keys.json"
MODAL_URL = "https://mauceri--aloepri-qwen3-modal-serve.modal.run"
API_KEY_PATH = "/home/cmauceri/.aloepri-api-key"
PYTHON = "/home/cmauceri/deepseek-harness-ws/venv/bin/python"
PORT = 8001

import subprocess, time, requests

with open(API_KEY_PATH) as f:
    API_KEY = f.read().strip()
repo = "/home/cmauceri/deepseek-harness-ws/Secretarius"

proc = subprocess.Popen(
    [PYTHON, f"{repo}/aloepri_modal/openai_proxy.py",
     "--keys", KEYS_PATH, "--url", MODAL_URL,
     "--api-key", API_KEY, "--port", str(PORT)],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("proxy démarré (PID", proc.pid, ")")
for _ in range(30):
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/v1/models", timeout=2).status_code == 200:
            print("proxy prêt : http://127.0.0.1:%d/v1/chat/completions" % PORT)
            break
    except Exception:
        time.sleep(0.5)

In [ ]:
# --- Client OpenAI officiel, pointé sur le proxy local ---
try:
    from openai import OpenAI
except ImportError:
    %pip install -q openai
    from openai import OpenAI

client = OpenAI(base_url=f"http://127.0.0.1:{PORT}/v1", api_key="local")

In [ ]:
# --- Premier appel (le 1er après une pause déclenche le cold start de
# Modal : ~1-3 min) ---
resp = client.chat.completions.create(
    model="qwen3-8b-obf",
    messages=[{"role": "user", "content": "Quelle est la capitale de la France ?"}],
    max_tokens=200,
)
print(resp.choices[0].message.content)
print("\n[usage]", resp.usage)

In [ ]:
# --- Multi-tours + modèle listé ---
print("modèles disponibles :", [m.id for m in client.models.list().data])

messages = [
    {"role": "system", "content": "Tu réponds en français, de façon concise."},
    {"role": "user", "content": "Quelle est la capitale de la France ?"},
    {"role": "assistant", "content": "Paris."},
    {"role": "user", "content": "Et celle de l'Italie ?"},
]
resp = client.chat.completions.create(model="qwen3-8b-obf", messages=messages, max_tokens=100)
print("réponse :", resp.choices[0].message.content)

## Notes

- **Cold start** : le premier appel après ~5 min d'inactivité relance le conteneur Modal (~1-3 min).
- **Arrêt** : `proc.kill()` dans la cellule de démarrage.
- **Posture stricte** : le proxy porte les clés ; ne pas exposer le port 8001 hors de la machine.
- `stream=True` n'est pas supporté par le proxy (greedy).